In [ ]:
census_api_key = "0f03d081293f18a970ca7a84d9b2889bcdee5f65"

# County Business Patterns, 1986-2023

Pulls CBP from the Census API (`https://api.census.gov/data/<year>/cbp`) using `census_api_key` from the cell above. Run the cells top to bottom; the last cell starts the download.

**Output:** one gzipped CSV per geography and year, e.g. `data/cbp_county_1986.csv.gz`. Read them with `pd.read_csv(path, dtype=str)` so FIPS and industry codes keep their leading zeros. Geography is in the `state` and `county` columns (plus `GEO_ID`).

**Things worth knowing**
- **Columns differ by era.** The industry code column is `SIC` for 1986-1997, then `NAICS1997` (1998-2002), `NAICS2002` (2003-2007), `NAICS2007` (2008-2011), `NAICS2012` (2012-2016) and `NAICS2017` (2017-2023). The column list is read from each year's API metadata, so every file has whatever that year publishes; nothing is renamed or harmonized.
- **Empty rows are dropped.** The API pads results with cells that have `ESTAB == 0` and zero employment and payroll (about 75-80% of rows before 2012). They are removed by default; pass `drop_empty=False` to keep them.
- **"Statewide" rows.** In 1986 they are county `999` ("Statewide <state>") in the county file. In 2023 they are instead separate rows in the *state* file, with a `GEO_ID` like `04000S0US01` and a `NAME` ending in "-- Statewide", and are absent from the county file. So 2023 county sums fall short of the state total by that amount; filter on `GEO_ID` if you sum the state file. I only checked 1986 and 2023, so check the years in between before relying on either pattern.
- **Counties the API doesn't publish.** In 2023 the API lists 253 counties for Texas (not 254) and 4 for Hawaii. County sums plus Statewide match the state total in 52 of 56 areas, and in the other four (TX, HI, AS, MP) the gap is exactly 1 establishment.
- **Territories.** The 2023 files include Puerto Rico and the island areas (state codes 60, 66, 69, 72, 78). The US row equals the sum of the 50 states and DC only.
- **Request limits.** The API caps each query at about 1M cells and, before 2012, prices it as if every county had every industry code in every size class. So the old years are fetched a few counties at a time (1998-2011: one county per request, roughly 3,000 requests per year). Measured with `workers=8`: a full 1986 county pull took about 5 minutes and 2023 about 1.5 minutes. For 1998-2011 I'd budget roughly 25-30 minutes per year (an estimate from a smaller test at ~2 requests/second), so several hours for the whole range. The default is 4 workers; `workers=8` is faster.
- **Resumable.** Each file is written to a `.part` file and renamed when complete. Re-running skips finished files, and any year that fails is listed at the end so you can re-run it.
- **Faster alternative for the old years:** the Census also publishes bulk county files at `https://www2.census.gov/programs-surveys/cbp/datasets/<year>/cbp<yy>co.zip` (6-16 MB each).

In [ ]:
import csv
import gzip
import re
import threading
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from math import prod
from pathlib import Path

import requests

API = "https://api.census.gov/data/{year}/cbp"
FIRST_YEAR, LAST_YEAR = 1986, 2023
CELL_LIMIT = 981_228  # the API rejects queries whose *estimated* size exceeds this
MAX_VARS = 50
RETRY_STATUS = {429, 500, 502, 503, 504}
# `for`/`in` already add `state`/`county` columns, so the geography variables are redundant.
GEO_VARS = {"ST", "STATE", "COUNTY", "MSA", "CBSA", "CSA", "CD", "ZIPCODE", "NATION"}
NON_DATA_VARS = {"for", "in", "ucgid"}
# Noise infusion (EMP_N, PAYANN_N, ... and their _F flags) starts with the 2007 data. The 2005 and
# 2006 metadata lists those variables anyway, but the API answers HTTP 500 if you request them.
NOISE_FIRST_YEAR = 2007
NOISE_COL = re.compile(r"_N(_F)?$")

In [ ]:
class CellLimitError(Exception):
    pass


class Client:
    def __init__(self, api_key):
        self.api_key = api_key
        self._local = threading.local()

    def _session(self):
        if not hasattr(self._local, "session"):
            self._local.session = requests.Session()
        return self._local.session

    def scrub(self, text):
        # requests puts the full URL, key included, in its exception messages
        return str(text).replace(self.api_key, "***")

    def fetch(self, url, params=None, tries=6):
        """GET `url` and return the parsed JSON ([] for 204), retrying transient failures."""
        params = {**(params or {}), "key": self.api_key}
        for attempt in range(tries):
            try:
                resp = self._session().get(url, params=params, timeout=300)
            except requests.RequestException as e:
                problem = self.scrub(e)
            else:
                if resp.status_code == 204:
                    return []
                if resp.status_code == 200:
                    try:
                        return resp.json()
                    except ValueError:
                        if "Invalid Key" in resp.text:
                            raise RuntimeError("The Census API rejected the API key as invalid.")
                        problem = f"non-JSON response {self.scrub(resp.text[:200])!r}"
                elif resp.status_code == 400 and "cell limit" in resp.text:
                    raise CellLimitError(resp.text)
                elif resp.status_code in RETRY_STATUS:
                    problem = f"HTTP {resp.status_code}"
                else:
                    raise RuntimeError(
                        f"HTTP {resp.status_code} from {url}: {self.scrub(resp.text[:300])}"
                    )
            if attempt < tries - 1:
                time.sleep(min(2**attempt, 60))
        raise RuntimeError(f"gave up on {url} after {tries} tries: {problem}")


def select_columns(variables, year):
    """Every data variable for the year plus the flag/label columns the API lists as attributes."""
    cols = []
    for name, spec in variables.items():
        if name in NON_DATA_VARS or name in GEO_VARS:
            continue
        cols.append(name)
        cols.extend(a for a in (spec.get("attributes") or "").split(",") if a)
    cols = list(dict.fromkeys(cols))
    if year < NOISE_FIRST_YEAR:
        cols = [c for c in cols if not NOISE_COL.search(c)]
    if len(cols) > MAX_VARS:
        raise RuntimeError(f"{len(cols)} columns requested, the API allows {MAX_VARS}")
    return cols

In [ ]:
class YearPuller:
    def __init__(self, client, year, drop_empty=True):
        self.client = client
        self.drop_empty = drop_empty
        self.url = API.format(year=year)
        variables = client.fetch(f"{self.url}/variables.json")["variables"]
        self.cols = select_columns(variables, year)
        # Before 2012 the API assumes every geography can return every industry code in every
        # size class (e.g. 1,106 SIC codes x 14 size classes) and prices the query accordingly.
        rows_per_geo = prod(variables.get(c, {}).get("limit") or 1 for c in self.cols)
        self.county_batch = (
            max(1, CELL_LIMIT // (rows_per_geo * (len(self.cols) + 2))) if rows_per_geo > 1 else None
        )

    def query(self, geo_for, geo_in=None):
        params = {"get": ",".join(self.cols), "for": geo_for}
        if geo_in:
            params["in"] = geo_in
        data = self.client.fetch(self.url, params)
        if not data:
            return None, []
        header, rows = data[0], data[1:]
        if self.drop_empty:
            estab = header.index("ESTAB")
            rows = [row for row in rows if row[estab] != "0"]
        return header, rows

    def codes(self, geo_for, geo_in=None):
        """FIPS codes for a geography level, from a one-row-per-unit query."""
        params = {"get": "ESTAB", "for": geo_for}
        if geo_in:
            params["in"] = geo_in
        data = self.client.fetch(self.url, params)
        return [row[-1] for row in data[1:]]

    def state(self, state):
        return self.query(f"state:{state}")

    def counties(self, state):
        """(header, rows) for every county in `state`, batching counties to stay under the cell limit."""
        counties = self.codes("county:*", f"state:{state}")
        size = max(1, self.county_batch or len(counties))
        pending = [counties[i : i + size] for i in range(0, len(counties), size)]
        header, rows = None, []
        while pending:
            batch = pending.pop()
            try:
                # Always list counties: before 2012 the API prices `county:*` as every county in the US.
                h, r = self.query(f"county:{','.join(batch)}", f"state:{state}")
            except CellLimitError:
                if len(batch) == 1:
                    raise
                self.county_batch = max(1, len(batch) // 2)
                pending += [batch[: len(batch) // 2], batch[len(batch) // 2 :]]
                continue
            header = header or h
            if h and h != header:
                raise RuntimeError(f"column mismatch in state {state}: {h} vs {header}")
            rows += r
        return header, rows

In [ ]:
def _write_results(path, results, expected):
    """Write (header, rows) results as they arrive; returns the number of data rows."""
    part = path.with_name(path.name + ".part")
    header, n_rows = None, 0
    with gzip.open(part, "wt", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        for done, (h, rows) in enumerate(results, 1):
            if h is None:
                continue
            if header is None:
                header = h
                writer.writerow(header)
            elif h != header:
                raise RuntimeError(f"column mismatch: {h} vs {header}")
            writer.writerows(rows)
            n_rows += len(rows)
            if expected > 10 and done % 10 == 0:
                print(f"    {done}/{expected} states", flush=True)
    part.replace(path)
    return n_rows


def _stream(pool, fn, items):
    """Yield fn(item) for each item as workers finish, without keeping finished results alive."""
    pending = {pool.submit(fn, item) for item in items}
    while pending:
        done, pending = wait(pending, return_when=FIRST_COMPLETED)
        for future in done:
            yield future.result()


def pull_year(client, year, geo, path, workers=4, states=None, drop_empty=True):
    """Download one geography for one year to `path`; returns the number of data rows."""
    puller = YearPuller(client, year, drop_empty)
    pool = ThreadPoolExecutor(max_workers=workers)
    try:
        if geo == "us":
            results, expected = [puller.query("us:*")], 1
        else:
            states = states or puller.codes("state:*")
            fetch = puller.counties if geo == "county" else puller.state
            results, expected = _stream(pool, fetch, states), len(states)
        return _write_results(path, results, expected)
    except BaseException:
        path.with_name(path.name + ".part").unlink(missing_ok=True)
        raise
    finally:
        pool.shutdown(wait=False, cancel_futures=True)


def pull_cbp(api_key, years=range(FIRST_YEAR, LAST_YEAR + 1), geos=("county",),
             out_dir="data", workers=4, states=None, overwrite=False, drop_empty=True):
    """Download CBP for each year and geography ("us", "state", "county") into `out_dir`.

    Finished files are skipped on re-runs unless `overwrite`. `states` limits county/state
    pulls to a list of 2-digit state FIPS codes. `drop_empty` discards the rows with ESTAB == 0
    that the API pads its results with. Returns a list of (year, geo, error) failures.
    """
    client = Client(api_key)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    client.fetch(API.format(year=LAST_YEAR), {"get": "ESTAB", "for": "us:*"}, tries=1)  # bad key? fail fast
    failures = []
    for year in years:
        for geo in geos:
            path = Path(out_dir) / f"cbp_{geo}_{year}.csv.gz"
            if path.exists() and not overwrite:
                print(f"{year} {geo}: {path.name} exists, skipping", flush=True)
                continue
            start = time.time()
            print(f"{year} {geo}: pulling...", flush=True)
            try:
                n_rows = pull_year(client, year, geo, path, workers, states, drop_empty)
            except Exception as e:
                print(f"{year} {geo}: FAILED: {client.scrub(e)}", flush=True)
                failures.append((year, geo, client.scrub(e)))
                continue
            print(f"{year} {geo}: {n_rows:,} rows -> {path} ({time.time() - start:.0f}s)", flush=True)
    if failures:
        print(f"\n{len(failures)} failed (re-run to retry them):", flush=True)
        for year, geo, err in failures:
            print(f"  {year} {geo}: {err}", flush=True)
    return failures

In [ ]:
# Quick check first (one small state, one year per era, all geographies, ~2 minutes):
# pull_cbp(census_api_key, years=[1986, 2005, 2023], geos=("us", "state", "county"),
#          states=["56"], out_dir="data_test")

# Full pull: every county, 1986-2023 -> data/cbp_county_<year>.csv.gz
# This takes hours (see the notes above). Finished years are skipped if you re-run it.
# Other options: geos=("us", "state", "county"), workers=8, states=[...], drop_empty=False
failures = pull_cbp(census_api_key)